# Mid-circuit measurement benchmark on IBM Quantum

This notebook benchmarks how well an IBM quantum processor performs the basic cycle of quantum error
correction - **measure an ancilla mid-circuit, then correct the data qubits based on the outcome** -
on every qubit of the chip that can act as an ancilla. It builds the circuits, prices them, submits
them, collects the results and draws a quality map of the device.

Section 0 explains the benchmark and the metrics; sections 1-7 run it:
**configure → backend → chains → plan & cost → submit → harvest → results**. Section 8 optionally
compares with an earlier campaign.

**Safe by default:** nothing is sent to IBM until you set `SUBMIT = True`. Until then the notebook still
produces results: it builds an Aer noise model from **the chosen device's own calibration** and runs the
very same circuits locally, so you can see roughly what the device would return, and check the whole
pipeline, before spending QPU time. Those results are written under the name `<device>_sim` and every
file is flagged `"simulated": true`, so they can never be mistaken for hardware data.

**No account?** Set `BACKEND_NAME = "noisy_simulator"`. The benchmark then runs on a synthetic heavy-hex
chip with uniform depolarizing noise, needing nothing but this package. Its error rates are arguments of
`SimBackend` with defaults in `qecbench.noise`, kept out of the configuration cell below.

**Requirements**
* `pip install -e ".[ibm]"` from the repository root (or keep the `sys.path` line of the next cell);
* for a real device, an IBM Quantum account saved once with
  `QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", instance="...", name="mcm-primitives")`,
  on a plan that includes the chosen device.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
DATA, FIGURES = ROOT / "data", ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qecbench import Chain, Direct
from qecbench.analysis import ancilla_scores, depth_summary, load_results
from qecbench.experiment import build_plan, harvest, print_plan, submit
from qecbench.layout import (ancilla_candidates, ancilla_coverage, grid_layout, min_circuits, pack,
                             qubit_load, select_chains, select_direct_chains)
from qecbench.lrqaoa import ideal_r, random_baseline
from qecbench.plotting import (plot_coverage, plot_depth_spread, plot_device_map, plot_groups,
                               plot_partition)

mpl.rcParams.update({"xtick.labelsize": 14, "ytick.labelsize": 14, "font.size": 14, "axes.linewidth": 2,
                     "axes.labelsize": 15, "lines.linewidth": 2, "legend.fontsize": 12})
%matplotlib inline

## 0. Background

### Why this benchmark

Quantum error correction repeats one cycle over and over: entangle an **ancilla** with a few **data
qubits**, **measure the ancilla in the middle of the circuit**, and act on the outcome with
**feed-forward** (a gate that is applied only if the measured bit is 1), then reuse the ancilla. Gate
and readout fidelities published by vendors do not capture that cycle: they leave out what happens to
the data qubits while the ancilla is read and the classical decision is taken, the reset of the
ancilla, and how all of this behaves when many ancillas are handled at once.

This notebook measures that cycle directly, with a task whose ideal answer is known exactly, on every
qubit of the device that can play the ancilla.

### The task: linear-ramp QAOA on an Ising chain

Take $n$ data qubits on a line and the Hamiltonian

$$H = \sum_{i=0}^{n-2} Z_i Z_{i+1}, \qquad E(x) = \sum_i z_i z_{i+1},\quad z_i = 1 - 2x_i \in \{+1,-1\}.$$

Its ground states are the two alternating bitstrings ($E_{\min} = -(n-1)$) and its worst states the
two uniform ones ($E_{\max} = n-1$). The circuit is the **linear-ramp QAOA** (LR-QAOA): start in
$|+\rangle^{\otimes n}$ and apply $p$ layers

$$e^{-i\gamma_k H}\;\text{then}\;\prod_i R_X(-2\beta_k)_i,\qquad
\gamma_k = \frac{k}{p}\Delta,\quad \beta_k = \frac{p-k+1}{p}\Delta,\quad k = 1..p .$$

The ramp mimics a slow anneal, so on a perfect device the result improves with depth $p$; on a real
device every extra layer also adds errors. Watching the quality versus $p$ separates the two.

### The two implementations of a $ZZ$ rotation

<img src="../figures/paper/1d-chain.jpeg" alt="1D chain: MCM and direct implementations" width="620">

*Left: the chain on the chip - data qubits $q_i$ (white) with an ancilla $a_i$ (yellow) between each
neighbouring pair; the vertical bars mark the $Z_iZ_{i+1}$ terms. Right: the two ways to apply one
such term. **MCM** routes it through the ancilla and a mid-circuit measurement; **direct** applies it
between two adjacent data qubits with no ancilla at all. Both realise the same
$\exp(-i\gamma Z Z)$, and comparing them is what this benchmark does.*

**MCM**, term by term: every $Z_iZ_j$ gets its own ancilla $a$, coupled to both data qubits.

1. The two CNOTs copy the parity $x_i \oplus x_j$ onto the ancilla.
2. $R_Z(2\gamma)$ on the ancilla then multiplies each branch by $e^{-i\gamma z_i z_j}$ - the wanted
   rotation $e^{-i\gamma Z_iZ_j}$ - but the ancilla is still entangled with the data.
3. Measuring the ancilla in the $X$ basis ($H$, then measure) disentangles it. Outcome $m=0$ leaves
   exactly $e^{-i\gamma Z_iZ_j}|\psi\rangle$; outcome $m=1$ leaves $Z_iZ_j\,e^{-i\gamma Z_iZ_j}|\psi\rangle$.
4. **Feed-forward** undoes the extra $Z_iZ_j$ when $m=1$, and the ancilla (left in $|m\rangle$) is
   returned to $|0\rangle$ for the next layer.

**Direct**, the reference: with the two data qubits adjacent, the same rotation is just
`CX, RZ(2γ), CX` on that coupler - one two-qubit rotation, no ancilla, no measurement, no
feed-forward.

Both vendors run the same logical circuits written with CZ gates, since
$\mathrm{CNOT}(d\to a) = H_a\,\mathrm{CZ}\,H_a$ and the inner Hadamards cancel: on the ancilla the MCM
gadget becomes `H, CZ(d_i,a), CZ(d_j,a), RX(2γ), measure`.

### Where it runs on the chip

A **chain** is a path of physical qubits `d0 - a0 - d1 - a1 - ... - d(n-1)`: data and ancillas
alternate, one ancilla per $ZZ$ term. The smallest chain, $n=2$, is the **triplet** `(d1, a, d2)`: one
ancilla, one mid-circuit measurement per layer. Triplets isolate the quality of each ancilla qubit;
longer chains add data qubits that are corrected by two neighbouring ancillas.

Many chains that share no qubit run **in the same circuit**, side by side, so a whole device is
covered in a handful of circuits. A qubit cannot be in two chains of one circuit, so the largest
number of chains any qubit belongs to (its **load**) is a lower bound on the circuits needed.

Three circuit **kinds** can be run on the same chains:

### The two implementations

The same Hamiltonian can be run two ways, and the benchmark is the comparison between them:

| what you run | qubits | how each $ZZ$ rotation is done | kind |
|---|---|---|---|
| **`Chain`** | $n$ data + $n-1$ ancillas, alternating | the gadget above: ancilla measured every layer, data corrected by feed-forward | `mcm` |
| **`Direct`** | $n$ data qubits, adjacent | applied straight to the bond, $\mathrm{RZZ}(2\gamma)$ - no ancilla, no measurement, no feed-forward | `direct` |

Both solve the **same $n$-spin Hamiltonian** at the same depth, so they share the noiseless
reference and their $r$ can be compared directly. Per layer and per bond, `mcm` pays one extra
mid-circuit measurement and its feed-forward; everything else is the same.

`Direct.from_chain(chain)` builds that counterpart. It cannot reuse the chain's own data qubits -
those sit two hops apart with the ancilla between them, and no native two-qubit gate reaches across -
so it takes $n$ *adjacent* qubits of the same path, keeping it on the same patch of the chip: a
triplet `(d1, a, d2)` gives the coupled pair `(d1, a)`. Both go into the same jobs, so they see the
same calibration, and the gap between them is what the measure-and-correct cycle costs.

### What is measured

From the sampled bitstrings of a chain's data qubits:

* **Approximation ratio** $\;r = \dfrac{E_{\max} - \langle E\rangle}{E_{\max} - E_{\min}}$ - 1 when only
  ground states are sampled, 0 for the worst states. Its shot-noise error is stored with it.
* **Random sampling** gives $\langle E\rangle = 0$, so $r_{\rm rand} = 1/2$ exactly, with a shot-noise
  spread $\sigma_{\rm rand} = 1/\big(2\sqrt{(n-1)S}\big)$ for $S$ shots.
* **Noiseless reference** $r_{\rm ideal}(p)$, computed exactly (the next cell plots it).
* **Normalised quality** $\;r_{\rm ovl} = \dfrac{r - r_{\rm rand}}{r_{\rm ideal} - r_{\rm rand}}$ - the
  fraction of the ideal improvement over random guessing that survives the device: 1 = noiseless,
  0 = indistinguishable from random. This is what the device maps show, and what is comparable across
  depths, chain lengths and devices. At small $p$ the denominator is small, so $r_{\rm ovl}$ is noisier
  there.

## 1. Configuration - the only cell you normally edit

* `N_DATA` - 2 benchmarks **triplets** (one ancilla per chain); 3 or more benchmarks longer chains.
* `DEPTHS`, `SHOTS`, `DELTA` - the LR-QAOA depths to sweep, shots per circuit and ramp amplitude $\Delta$.
* `DIRECT_REFERENCE` - also run the ancilla-free `Direct` implementation on the same qubits, in the
  same jobs (table in section 0). It roughly doubles the cost and is what makes the result
  interpretable.
* `ALLOWED_QUBITS` / `ANCILLAS` - restrict the experiment to part of the chip, or restrict which qubits
  are benchmarked as ancillas.
* `BY_CALIBRATION` - when several neighbour choices pack equally well, prefer the one with the lowest
  calibrated error.
* `BUFFER`, `MAX_PER_BATCH` - how densely chains are packed into circuits (see section 4). Density can
  change the result, so keep these fixed across depths and devices you want to compare.
* `MAX_CIRCUITS_PER_JOB` - IBM's classical control electronics load a whole job at once; jobs with too
  many feed-forward operations fail (error 6073). 8 circuits per job has been safe.
* `DYNAMICAL_DECOUPLING` - pulse sequences on idle qubits; off by default so the raw cycle is measured.

In [ ]:
BACKEND_NAME = "ibm_phoenix"        # any IBM device, or "noisy_simulator" to run without an account
ACCOUNT      = "mcm-primitives"     # name of the saved IBM account; None for the default one
N_DATA       = 2                    # 2 = triplets; 3, 4, ... = longer chains
DEPTHS       = [3, 6, 9, 12]
SHOTS        = 500
DELTA        = 0.5
DIRECT_REFERENCE = True             # also run the ancilla-free Direct version on the same qubits

ALLOWED_QUBITS = None               # None = whole chip, e.g. list(range(40))
ANCILLAS       = None               # restrict which qubits are benchmarked as ancillas
BY_CALIBRATION = True               # prefer the best-calibrated neighbours when packing is a tie
BUFFER         = False              # True: chains sharing a circuit are not even neighbours
MAX_PER_BATCH  = None               # cap chains per circuit (None = as many as fit)

OPTIMIZATION_LEVEL   = 1
DYNAMICAL_DECOUPLING = False
MAX_CIRCUITS_PER_JOB = 8            # larger jobs risk error 6073 (control-electronics memory)

SUBMIT = False                      # True actually submits the jobs and consumes QPU time

What a perfect device would give - no hardware involved. The grey band is where random guessing
lands with the number of shots configured below; a measurement inside it carries no signal.

In [ ]:
ps = list(range(1, 21))
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for n in (2, 3, 5):
    chain = Chain(range(2 * n - 1))
    ax.plot(ps, [ideal_r(chain, p, DELTA) for p in ps], "o-", ms=4, label=f"noiseless, n_data = {n}")
sigma = random_baseline(Chain((0, 1, 2)), SHOTS)[1]
ax.axhspan(0.5 - 3 * sigma, 0.5 + 3 * sigma, color="gray", alpha=0.3, lw=0,
           label=f"random guessing ±3σ (triplet, {SHOTS} shots)")
ax.set_xlabel("LR-QAOA depth $p$")
ax.set_ylabel("$r$")
ax.set_xticks([1, 5, 10, 15, 20])
ax.legend(frameon=False, fontsize=10)
triplet = Chain((0, 1, 2))
print("noiseless r of a triplet at the configured depths:",
      {p: round(ideal_r(triplet, p, DELTA), 3) for p in DEPTHS})

## 2. Backend and calibration

Connects to the device and saves a timestamped calibration snapshot to `data/calibration/`, because
calibrations drift and only the snapshot taken next to a run can be compared with it. The connection
is needed either way: with `SUBMIT = False` that same calibration is what the local noise model is
built from.

IBM calibrates **two different readouts**: `measure`, the final readout (what the data qubits get at
the end), and `measure_2`, the **mid-circuit** readout actually used for the ancillas. Their errors
differ qubit by qubit, so the snapshot keeps both, and the per-chain error estimate charges the
ancillas the mid-circuit one. The last printed line confirms the device supports mid-circuit
measurement at all.

In [ ]:
from qecbench.backends import IBMBackend, SimBackend

if BACKEND_NAME == "noisy_simulator":
    # No account, no device: a synthetic heavy-hex chip with the same depolarizing noise everywhere.
    # Defaults live in qecbench.noise; override them here if you want, e.g. SimBackend(p_2q=0.01).
    backend, cal = SimBackend(seed=1), None
    print(f"{backend.name}: {backend.coupling_graph().number_of_nodes()} qubits, "
          f"{backend.noise_description}")
else:
    # With SUBMIT = False this is still the real device object, but the plan is executed on Aer under
    # a noise model built from its calibration, and written under the name "<device>_sim".
    backend = IBMBackend(BACKEND_NAME, account=ACCOUNT, optimization_level=OPTIMIZATION_LEVEL,
                         dynamical_decoupling=DYNAMICAL_DECOUPLING,
                         max_circuits_per_job=MAX_CIRCUITS_PER_JOB, local=not SUBMIT, seed=1)
    status = getattr(backend.backend, "status", lambda: None)()
    print(f"{backend.name}: {backend.backend.num_qubits} qubits"
          + (f", {status.status_msg}, {status.pending_jobs} jobs queued" if status else "")
          + f", mid-circuit measurement: {backend.supports_mcm()}")
    if backend.simulated:
        print(f"local simulation - {backend.noise_description}")
    cal = backend.calibration(save_dir=DATA / "calibration")

if backend.simulated:
    ALLOWED_QUBITS = ALLOWED_QUBITS or list(range(20))        # noisy simulation: keep it small

## 3. Choose the chains

Every qubit with at least two neighbours can be an ancilla. The selection gives each of them a turn:

* **Triplets** (`N_DATA = 2`): each candidate is the ancilla of **exactly one** triplet. A qubit with
  three neighbours has three possible neighbour pairs; the pair is chosen so that no qubit ends up in
  too many triplets, which keeps the number of circuits at its lower bound (on IBM's heavy-hex chips,
  typically 4 circuits for the whole device).
* **Longer chains**: paths are grown until every candidate is the ancilla of **at least one** chain; a
  few are covered twice. Qubits no chain of this length can reach are reported.

The printout gives the coverage, the parallelism against its lower bound and, with a calibration
loaded, the predicted error of the best and worst chains.

The map shows what the numbers cannot: **which couplers the gadgets will actually use**. Filled nodes
are benchmarked as ancillas, hollow ones only carry data, pale ones are untouched, and thick edges are
the couplers in use. With a calibration the ancillas are shaded by their predicted error per layer, so
the weak spots are visible before anything is submitted; without one they share a single colour, since
the roles are categories rather than a scale.

In [ ]:
G = backend.coupling_graph()
pos = grid_layout(G)
cost = (lambda chain: backend.error_budget(chain, cal)) if (cal and BY_CALIBRATION) else None
chains = select_chains(G, N_DATA, allowed_qubits=ALLOWED_QUBITS, ancillas=ANCILLAS, cost=cost,
                       buffer=BUFFER)
assert chains, "no chain fits - widen ALLOWED_QUBITS / ANCILLAS, or lower N_DATA"

region = set(G.nodes) if ALLOWED_QUBITS is None else set(ALLOWED_QUBITS)
cands, cover, load = ancilla_candidates(G, ALLOWED_QUBITS, ANCILLAS), ancilla_coverage(chains), qubit_load(chains)
batches = pack(chains, G, buffer=BUFFER)
missed = sorted(set(cands) - set(cover))

print(f"device      : {backend.name}, {G.number_of_nodes()} qubits, {G.number_of_edges()} couplers"
      + ("" if ALLOWED_QUBITS is None else f"   (restricted to {len(region)} qubits)"))
print(f"chains      : {len(chains)} of {N_DATA} data qubits, e.g. {chains[0].qubits} "
      f"(data {chains[0].data_qubits}, ancilla {chains[0].ancillas})")
print(f"coverage    : {len(cover)}/{len(cands)} ancilla-capable qubits benchmarked"
      + (f", missed {missed}" if missed else "")
      + (f", {max(cover.values())}x for the most-used" if max(cover.values()) > 1 else ", once each"))
print(f"qubits used : {len(load)}/{len(region)}   ({len(cover)} as ancilla, "
      f"{len(load) - len(cover)} data only, {len(region) - len(load)} untouched)")
print(f"parallelism : {len(batches)} circuits per depth (lower bound {min_circuits(chains)}), "
      f"{'/'.join(str(len(b)) for b in batches)} chains running at once")

if cal:
    budget = {c: backend.error_budget(c, cal) for c in chains}
    rank = sorted(budget, key=budget.get)
    print(f"calibration : predicted error per layer, median {np.median(list(budget.values())):.4f}"
          f"   best {rank[0].qubits} {budget[rank[0]]:.4f}   worst {rank[-1].qubits} {budget[rank[-1]]:.4f}")
    flagged = backend.flag_instances(chains, cal)
    print(f"              {len(flagged)} chain(s) touch hardware outside the usual thresholds"
          + (f", e.g. {flagged[0][0].qubits}: {flagged[0][1]}" if flagged else ""))

# the same Hamiltonian without the ancilla, on n_data adjacent qubits of each chain's own path
if DIRECT_REFERENCE:
    direct = sorted({Direct.from_chain(c) for c in chains})   # neighbouring chains can share a pair
    instances = chains + direct
    print(f"instances   : {len(instances)} = {len(chains)} mcm + {len(direct)} direct"
          + (f"   ({len(chains) - len(direct)} chain(s) share a direct counterpart)"
             if len(direct) < len(chains) else ""))
else:
    instances = list(chains)
    print(f"instances   : {len(instances)} = {len(chains)} mcm, no direct reference")

NODE_SIZE, FONT_SIZE, MAP_FIGSIZE = (260, 6, (14, 13)) if G.number_of_nodes() > 60 else (500, 9, (8, 7))
fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
plot_coverage(G, chains, pos=pos, ax=ax, node_size=NODE_SIZE, font_size=FONT_SIZE,
              cost=budget if cal else None,
              title=f"{backend.name} - {len(chains)} chains of {N_DATA} data qubits")
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_coverage.pdf", bbox_inches="tight")

## 4. Plan and cost - before anything is sent

`build_plan` packs the chains into circuits, builds every circuit for every depth and kind, and
**transpiles it with the physical qubits fixed**. If the transpiler had to insert any gate on a coupler
outside the chains (routing), the chain would no longer be the chain it claims to be, and the plan is
refused.

**How IBM bills:** QPU time. Each shot costs the circuit's duration plus a fixed wait between shots;
the estimate schedules every circuit with the device's own gate and readout durations. One number is
not published: how long the classical feed-forward takes. The estimate assumes 1.9 µs per conditional
operation per layer - an **unverified assumption**, so read the QPU seconds as an order of magnitude.
IBM's own usage report after submission is authoritative.

**Why packing density matters:** if the control electronics process the feed-forward operations of
one layer one after another, the data qubits of a crowded circuit wait longer at every layer and
decohere more. `BUFFER = True` (chains in one circuit are never neighbours, suppressing measurement
cross-talk) and `MAX_PER_BATCH` both trade more circuits for less of that.

**What to check in the printout below**

* *Instances* - both kinds are listed if `DIRECT_REFERENCE = True`, with how many run in parallel per
  circuit; `5 + 5 + 5` means three circuits, five chains each.
* *Circuits* and *Jobs* - the whole sweep, and how it is split into submissions.
* The table - QPU seconds per depth and kind, and the total in seconds and dollars. Nothing has been
  sent at this point; if the total is more than you want to spend, restrict `ALLOWED_QUBITS`, shorten
  `DEPTHS` or lower `SHOTS` and re-run this cell.

In [ ]:
plan = build_plan(backend, instances, depths=DEPTHS, shots=SHOTS, delta=DELTA,
                  buffer=BUFFER, max_per_batch=MAX_PER_BATCH)
print_plan(plan, backend)

**The circuits of one depth.** Each panel is one circuit: coloured paths are the chains that run in
it simultaneously, filled nodes their ancillas, hollow nodes their data qubits. Every chain appears in
exactly one panel, and the same partition is reused at every depth, so a chain always shares its
circuit with the same neighbours.

In [ ]:
batches = plan["batches"]["mcm"]
fig, axes = plot_partition(G, batches, pos=pos, ncols=2, node_size=NODE_SIZE // 2,
                           title=f"{backend.name} - the {len(batches)} mcm circuits of one depth")
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_partition.pdf", bbox_inches="tight")

## 5. Inspect, then submit

The first cell draws the logical circuit of one chain at $p = 1$ for each kind - the gadget of
section 0 (`H`, two `CZ`, `RX(2γ)` and the measurement on the ancilla, the conditional `If` block
with the corrections, then the `RX(-2β)` mixer and the final readout). It costs nothing.

The second cell runs the plan. With `SUBMIT = True` it sends the jobs to the device and prints their ids;
with `SUBMIT = False` it simulates them locally under the device's calibration - same circuits, same
bookkeeping, no QPU time. Either way a manifest is written after every job, so an interrupted run still
knows what was sent, and the results are harvested the same way in section 6.

Nothing happens until you execute this cell, so to see only the plan and its cost, just stop above it.

In [ ]:
from qecbench.circuits import build_dynamic

for kind, batches_of_kind in plan["batches"].items():
    print(f"--- {kind}, one instance, p = 1 (logical circuit before transpilation)")
    print(build_dynamic(batches_of_kind[0][:1], 1, DELTA).draw(output="text", fold=120))

In [ ]:
manifest_path = submit(plan, backend, manifest_dir=DATA / "manifests")

## 6. Harvest

Submission writes a **manifest** (`data/manifests/<backend>/`): which chains sit in which circuit of
which job. Harvesting needs only that file, so it can happen hours or days later, in a new session. The
cell takes the newest real manifest of this device (paste another path to harvest an older run),
downloads finished jobs, skips unfinished ones, and writes one result file per chain and depth.
Running it again only adds what is new.

Results are filed as `data/results/<backend>/<structure>/<kind>/`, e.g.
`ibm_phoenix/chain/mcm/` and `ibm_phoenix/chain/direct/`, so a long campaign stays navigable and a
surface-code run later lands beside the chains rather than in the same heap.

**One file per run**, not per chain: each holds a `run` header (manifest, shots, depths, whether it was
simulated and under what noise) and a `results` list with every chain and depth. Harvesting the same
manifest again merges into that file, so jobs that finish later are simply added.

Each result file stores the raw counts of the chain's data qubits, $r$ with its error, the random and
noiseless references and $r_{\rm ovl}$, plus the job id and circuit it came from.

In [ ]:
manifests = sorted((p for p in (DATA / "manifests").rglob(f"*_{backend.name}_*.json")
                    if not p.name.endswith("_memory.json")), key=lambda p: p.name)   # memory runs: their own notebook
assert manifests, "no manifest yet - run the submit cell of section 5 first"
manifest_path = manifests[-1]
print("harvesting", manifest_path.name)
saved = harvest(manifest_path, backend, data_dir=DATA / "results")

## 7. Results

**Depth sweep.** Left: $r$ versus $p$ - one faint line per instance, the median (points) and the
inter-quartile band, against the noiseless curve (dashed) and the random-guessing band (grey). Right:
the same data as $r_{\rm ovl}$, which divides out the ramp so that a perfect device would sit at 1 for
every $p$. A steady decline of $r_{\rm ovl}$ with $p$ is the cost of each extra layer.

Blue is the measurement-based chain, orange the direct implementation on the same qubits (if it was
run). **The gap between them is the result**: the part of the decay that belongs to the mid-circuit
measurement and the feed-forward rather than to the gates and idling of those qubits.

In [ ]:
MAP_KIND = "mcm"
results = load_results(DATA / "results", backend.name, kind=MAP_KIND, n_data=N_DATA)
assert results, f"no {MAP_KIND} results for {backend.name} yet - submit and harvest first"
# the direct implementation solves the same Hamiltonian, so it has the same number of data qubits
direct = load_results(DATA / "results", backend.name, kind="direct", n_data=N_DATA)
print(f"{len(results)} chains and {len(direct)} direct instances with data")
print(f"{'p':>4} {'median r':>9} {'IQR':>17} {'median r_ovl':>13} {'n':>5}")
r_sum, o_sum = depth_summary(results, "r"), depth_summary(results, "r_ovl")
for p in r_sum:
    print(f"{p:>4} {r_sum[p][0]:>9.3f}  [{r_sum[p][1]:.3f}, {r_sum[p][2]:.3f}] {o_sum[p][0]:>13.3f} {r_sum[p][3]:>5}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_depth_spread(results, "r", ax=axes[0], label="chains (mcm)")
plot_depth_spread(results, "r_ovl", ax=axes[1], label="chains (mcm)")
if direct:
    plot_depth_spread(direct, "r", ax=axes[0], color="#b8560f", label="direct", reference=False)
    plot_depth_spread(direct, "r_ovl", ax=axes[1], color="#b8560f", label="direct")
    d_sum = depth_summary(direct, "r_ovl")
    print("\nwhat the measure-and-correct cycle costs, in r_ovl:")
    print(f"{'p':>4} {'mcm':>8} {'direct':>8} {'gap':>8}")
    for p in sorted(set(o_sum) & set(d_sum)):
        print(f"{p:>4} {o_sum[p][0]:>8.3f} {d_sum[p][0]:>8.3f} {d_sum[p][0] - o_sum[p][0]:>+8.3f}")
axes[1].set_ylim(-0.2, 1.1)
fig.tight_layout()
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_{MAP_KIND}_depth.pdf", bbox_inches="tight")

**Device map.** Every ancilla qubit coloured by $r_{\rm ovl}$ of its chain at one depth (the best
chain, if a qubit was the ancilla of several). Grey qubits were not benchmarked as ancillas. Dark
spots are the qubits that handle mid-circuit measurement and feed-forward worst - the ones to avoid as
syndrome qubits. The lists below name the best and worst chains.

In [ ]:
PLOT_DEPTH = DEPTHS[1] if len(DEPTHS) > 1 else DEPTHS[0]
scores = ancilla_scores(results, depth=PLOT_DEPTH, value="r_ovl")
fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
plot_device_map(G, scores, pos=pos, ax=ax, vmin=0, vmax=1, node_size=NODE_SIZE, font_size=FONT_SIZE,
                label=r"$r_{\rm ovl} = (r - r_{\rm rand}) / (r_{\rm ideal} - r_{\rm rand})$",
                title=f"{backend.name} - quality per ancilla, n_data = {N_DATA}, p = {PLOT_DEPTH}")
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_{MAP_KIND}_map.pdf", bbox_inches="tight")

ranked = sorted(((by[PLOT_DEPTH]["r_ovl"], inst) for inst, by in results.items() if PLOT_DEPTH in by),
                key=lambda x: -x[0])
print("best :", [(str(i.qubits), round(v, 3)) for v, i in ranked[:5]])
print("worst:", [(str(i.qubits), round(v, 3)) for v, i in ranked[-5:]])

## 8. Compare with an earlier campaign (optional)

Result files from any earlier run of this benchmark load the same way - including triplet files named
`*_tri_<d1>_<a>_<d2>_*` written before this package existed. Every number is recomputed from the stored
counts, so old and new runs are compared on identical footing.

Point `EARLIER_DATA` at the folder holding those files and, optionally, `EARLIER_MANIFEST` at the
manifest of one submission to keep only its jobs. The default is the triplet campaign of 2026-09-11 on
`ibm_phoenix`; the cell is skipped if the folder does not exist.

Files of any kind load, including kinds this package no longer builds: that campaign also ran
`mcm_reset`, a variant that returned the ancilla to $|0\rangle$ with a `reset` instruction instead of
the conditional $X$. Drop it from `EARLIER_KINDS` if you only want the benchmark itself.

In [ ]:
EARLIER_DATA = Path("path/to/earlier/Data")   # folder of earlier result files
EARLIER_BACKEND = "ibm_phoenix"
EARLIER_MANIFEST = EARLIER_DATA / "manifests" / "20260911_102009_ibm_phoenix_mcm_triples.json"  # None = all runs
EARLIER_KINDS = ("mcm", "mcm_reset")   # mcm_reset: earlier ancilla-recycling variant, still readable

if EARLIER_DATA.exists():
    earlier = {kind: load_results(EARLIER_DATA, EARLIER_BACKEND, kind=kind, manifest_path=EARLIER_MANIFEST)
               for kind in EARLIER_KINDS}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for (kind, res), colour in zip(earlier.items(), ("#2f6f9f", "#b8560f")):
        if res:
            plot_depth_spread(res, "r", ax=axes[0], color=colour, label=f"{kind} triplets", reference=kind == "mcm")
            plot_depth_spread(res, "r_ovl", ax=axes[1], color=colour, label=f"{kind} triplets")
            print(f"{kind:>9}: {len(res)} triplets, median r_ovl by depth",
                  {p: round(v[0], 3) for p, v in depth_summary(res, "r_ovl").items()})
    axes[0].set_title(f"{EARLIER_BACKEND}, {EARLIER_MANIFEST.name if EARLIER_MANIFEST else 'all runs'}", fontsize=11)
    fig.tight_layout()
else:
    print("no earlier results found at", EARLIER_DATA)